# DFT Fermi surfaces from JARVIS-DFT structures (VASP + IFermi)

For a quantitative, DFT-quality Fermi surface we need band energies on a uniform
Brillouin-zone grid. JARVIS-DFT provides the relaxed **structures** (and many
properties), but not 3D eigenvalue grids, so the workflow is:

1. pull the relaxed structure from JARVIS-DFT,
2. run one **static VASP** calculation on a dense, non-symmetry-reduced k-mesh
   (`ISYM = 0`), and
3. extract the Fermi surface from `vasprun.xml` with IFermi via
   `fermisurf.vasp_fermi`.

Compare the shapes with https://www.hzdr.de/projects/fermisur/ . For a quick,
DFT-free surface, see the companion `slakonet_fermi_surface.ipynb`.

## 1. Structure from JARVIS-DFT

In [ ]:
import sys
sys.path.insert(0, "..")

from slakonet.optim import get_atoms   # convenient JARVIS-DFT structure loader

jid = "JVASP-816"   # aluminum (fcc); e.g. Nb JVASP-14818, Mg JVASP-194201, Ti JVASP-194005
atoms = get_atoms(jid)[0]
atoms.write_poscar("POSCAR")
print(atoms.composition.reduced_formula, jid, "-> wrote POSCAR")

## 2. VASP inputs for a Fermi-surface run

Use the POSCAR above with a dense, full (unreduced) Monkhorst-Pack mesh and a static
self-consistent run. The key settings:

```
# INCAR
ISYM = 0        # full BZ, no symmetry reduction (IFermi expands the mesh itself,
                # but a full grid is the simplest and most robust)
ISMEAR = 1 ; SIGMA = 0.2
NEDOS = 3001
LWAVE = .FALSE. ; LCHARG = .FALSE.
```

```
# KPOINTS (Gamma-centred, dense)
Automatic
0
Gamma
16 16 16
0 0 0
```

Run `vasp_std`; this produces the `vasprun.xml` the next cell reads.

## 3. Fermi surface from vasprun.xml

In [ ]:
import os
from fermisurf.vasp_fermi import build_surface
from ifermi.plot import FermiSurfacePlotter, save_plot

if os.path.exists("vasprun.xml"):
    fs = build_surface("vasprun.xml", interpolation_factor=5)
    fig = FermiSurfacePlotter(fs).get_plot(plot_type="plotly")
    fig.show()
    save_plot(fig, f"{jid}_dft_fermi.html")
    print(f"wrote {jid}_dft_fermi.html")
else:
    print("Run VASP (step 2) first, then re-run this cell with vasprun.xml present.")
    print("Equivalent CLI: python -m fermisurf.vasp_fermi --vasprun vasprun.xml --out fs.html")

IFermi handles the Fourier interpolation, symmetry, and the Wigner-Seitz Brillouin
zone, so this route gives the quantitative DFT surface (the slakonet notebook gives the
same topology from a tight-binding model, with no DFT run).